# Practice Problems Solutions: Introduction to AI-Powered Data Engineering

**Module:** Introduction to AI-Powered Data Engineering  
**Purpose:** Solution key for self-assessment

---

## Setup

Run this cell to ensure all required packages are available.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import os

print(f"pandas version: {pd.__version__}")
print(f"Current working directory: {os.getcwd()}")
print("Setup complete!")

---

## Part A: Conceptual Understanding

### Problem 1: Lifecycle Stage Identification

**Scenario:** Fintech company using ML for fraud detection.

| Description | Lifecycle Stage | Explanation |
|-------------|-----------------|-------------|
| A. Credit card swipes are captured from point-of-sale terminals | **Ingestion** | Data is being collected from source systems |
| B. Raw transaction records are stored in a cloud data lake | **Storage** | Data is being persisted for retention and access |
| C. Transaction amounts converted to USD and enriched with merchant codes | **Transformation** | Raw data is being cleaned, standardized, and enriched |
| D. Fraud detection model receives batch of cleaned transactions hourly | **Serving** | Curated data is delivered to downstream ML consumer |
| E. Alert fires when model precision drops below 85% | **Monitoring** | System observes performance and triggers alerts |

### Problem 2: Batch vs. Streaming Classification

**1. Daily sales reports**
- **Processing type:** Batch  
- **Justification:** Reports summarize historical data at scheduled intervals; no real-time requirement exists.

**2. Real-time fraud alerts**
- **Processing type:** Streaming  
- **Justification:** Fraud must be detected and blocked within milliseconds before the transaction completes.

**3. Training a recommendation model on 6 months of data**
- **Processing type:** Batch  
- **Justification:** Model training is a periodic, compute-intensive job processing large historical datasets.

---

## Part B: Python Fundamentals for Data Engineering

### Problem 3: Working with File Paths - SOLUTION

In [ ]:
from pathlib import Path

# SOLUTION: Create a Path object pointing to the customers.csv file
data_file = Path("../datasets/customers.csv")

# SOLUTION: Check if the file exists and print the result
file_exists = data_file.exists()
print(f"File exists: {file_exists}")

# SOLUTION: If the file exists, print its size in bytes
if file_exists:
    file_size = data_file.stat().st_size
    print(f"File size: {file_size} bytes")

# BONUS: What happens when a file doesn't exist?
fake_file = Path("data/nonexistent.csv")
print(f"\nFake file exists: {fake_file.exists()}")
print("Note: .exists() returns False for non-existent files instead of raising an error")

In [ ]:
# Additional demonstration: useful Path methods
data_file = Path("../datasets/customers.csv")

print(f"File name: {data_file.name}")
print(f"File stem (without extension): {data_file.stem}")
print(f"File extension: {data_file.suffix}")
print(f"Parent directory: {data_file.parent}")
print(f"Absolute path: {data_file.resolve()}")

### Problem 4: Reading and Inspecting CSV Data - SOLUTION

In [ ]:
import pandas as pd
from pathlib import Path

def inspect_csv(file_path: str) -> dict:
    """
    Load a CSV file and return basic statistics about it.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        Dictionary with row_count, column_count, and column_names
    """
    # SOLUTION: Load the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    
    # SOLUTION: Create and return a dictionary with the required info
    result = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "column_names": list(df.columns)
    }
    
    return result

# Test the function
stats = inspect_csv("../datasets/customers.csv")
print("CSV Inspection Results:")
print(f"  Row count: {stats['row_count']}")
print(f"  Column count: {stats['column_count']}")
print(f"  Columns: {stats['column_names']}")

### Problem 5: Basic Error Handling - SOLUTION

In [ ]:
from pathlib import Path

def safe_read_file(file_path: str) -> dict:
    """
    Safely attempt to read a file and return status information.
    
    Args:
        file_path: Path to the file
        
    Returns:
        Dictionary with success status, file_size or error message
    """
    result = {
        "success": False,
        "file_path": file_path,
        "file_size_bytes": None,
        "error": None
    }
    
    try:
        path = Path(file_path)
        # SOLUTION: Get file size and update result
        result["file_size_bytes"] = path.stat().st_size
        result["success"] = True
    except FileNotFoundError:
        # SOLUTION: Handle file not found
        result["error"] = f"File not found: {file_path}"
    except Exception as e:
        # SOLUTION: Handle any other exception
        result["error"] = f"Unexpected error: {str(e)}"
    
    return result

# Test with a file that exists
print("Test 1 - File that exists:")
result1 = safe_read_file("../datasets/customers.csv")
for key, value in result1.items():
    print(f"  {key}: {value}")

print("\nTest 2 - File that doesn't exist:")
result2 = safe_read_file("nonexistent_file.csv")
for key, value in result2.items():
    print(f"  {key}: {value}")

### Problem 6: Understanding Cloud Upload Patterns - SOLUTION

**a) Why do we check if the local file exists before attempting to upload?**

> To fail fast with a clear error message. If we skip this check, the upload call would fail with a less clear error from the cloud SDK. Validating inputs early makes debugging easier and provides better user feedback.

**b) What is the purpose of step 4 (verifying upload)?**

> To confirm the upload actually succeeded. Network issues, permission problems, or cloud service errors could cause silent failures. By fetching metadata (like file size) from the remote object, we can verify the file arrived intact and matches what we sent.

**c) Why organize cloud storage with path-like structures (e.g., `raw/customers/2025/01/customers.csv`)?**

> Several reasons:
> - **Organization:** Makes it easy to find files by date or category
> - **Partitioning:** Query engines (Spark, Athena) can skip irrelevant folders, improving performance
> - **Lifecycle management:** Easy to set retention policies (e.g., delete files older than 90 days)
> - **Access control:** Can set permissions at the folder level

### Problem 7: Timestamps and Data Organization - SOLUTION

In [ ]:
from datetime import datetime

def generate_upload_path(base_folder: str, filename: str) -> str:
    """
    Generate a cloud storage path organized by date.
    
    Example output: "raw/customers/2025/01/23/customers.csv"
    
    Args:
        base_folder: Base folder path (e.g., "raw/customers")
        filename: Name of the file (e.g., "customers.csv")
        
    Returns:
        Full path with date components
    """
    # SOLUTION: Get current date
    now = datetime.now()
    
    # SOLUTION: Extract year, month, day as zero-padded strings
    # Method 1: Using strftime
    year = now.strftime("%Y")
    month = now.strftime("%m")
    day = now.strftime("%d")
    
    # Alternative Method 2: Using f-strings with zero-padding
    # year = str(now.year)
    # month = f"{now.month:02d}"
    # day = f"{now.day:02d}"
    
    # SOLUTION: Construct and return the full path
    full_path = f"{base_folder}/{year}/{month}/{day}/{filename}"
    
    return full_path

# Test the function
path = generate_upload_path("raw/customers", "customers.csv")
print(f"Generated path: {path}")

# Additional examples
print(f"\nMore examples:")
print(f"  {generate_upload_path('bronze/transactions', 'sales.parquet')}")
print(f"  {generate_upload_path('landing/events', 'clickstream.json')}")

### Problem 8: Putting It Together - Mini Pipeline - SOLUTION

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

def prepare_for_upload(local_path: str, destination_bucket: str) -> dict:
    """
    Prepare a local CSV file for cloud upload.
    
    This function:
    1. Validates the file exists
    2. Inspects the data
    3. Generates the destination path
    4. Returns a summary ready for upload
    
    Args:
        local_path: Path to the local CSV file
        destination_bucket: Name of the cloud storage bucket
        
    Returns:
        Dictionary with upload details or error information
    """
    result = {
        "ready_for_upload": False,
        "local_path": local_path,
        "destination_bucket": destination_bucket,
        "destination_key": None,
        "file_size_bytes": None,
        "row_count": None,
        "timestamp": datetime.now().isoformat(),
        "error": None
    }
    
    try:
        # Step 1: Check if file exists
        path = Path(local_path)
        if not path.exists():
            result["error"] = f"File not found: {local_path}"
            return result
        
        # Step 2: Get file size
        result["file_size_bytes"] = path.stat().st_size
        
        # Step 3: Load and get row count (for CSV files)
        df = pd.read_csv(local_path)
        result["row_count"] = len(df)
        
        # Step 4: Generate destination key using the filename and current date
        now = datetime.now()
        filename = path.name
        result["destination_key"] = f"raw/data/{now.strftime('%Y/%m/%d')}/{filename}"
        
        # Step 5: Set ready_for_upload = True if all checks pass
        result["ready_for_upload"] = True
        
    except pd.errors.EmptyDataError:
        result["error"] = "CSV file is empty"
    except pd.errors.ParserError as e:
        result["error"] = f"CSV parsing error: {str(e)}"
    except Exception as e:
        result["error"] = f"Unexpected error: {str(e)}"
    
    return result

# Test the function with the actual dataset
print("Testing prepare_for_upload function:")
print("=" * 50)

result = prepare_for_upload("../datasets/customers.csv", "my-data-bucket")
for key, value in result.items():
    print(f"{key}: {value}")

print("\n" + "=" * 50)
print("Testing with non-existent file:")
print("=" * 50)

result2 = prepare_for_upload("fake_file.csv", "my-data-bucket")
for key, value in result2.items():
    print(f"{key}: {value}")

---

## Part C: Concept Check (Multiple Choice) - SOLUTIONS

### Question 1
**Why is data engineering considered foundational to AI systems?**

**Correct Answer: B** — Models cannot function without reliable data pipelines delivering quality data

*Explanation:* Data engineering provides the infrastructure that feeds AI models. Without reliable pipelines, even well-trained models fail because they don't receive the data they need.

---

### Question 2
**Which statement best describes the relationship between batch and streaming?**

**Correct Answer: C** — Most production systems combine both patterns for different use cases

*Explanation:* Real-world systems typically use batch for historical analysis and model training, while using streaming for real-time features and immediate responses. The patterns are complementary.

---

### Question 3
**The AI data lifecycle is described as "iterative" because:**

**Correct Answer: B** — Monitoring insights feed back into earlier stages, enabling continuous improvement

*Explanation:* The lifecycle is a loop, not a linear process. Monitoring detects issues (data drift, quality problems, performance degradation) that inform changes to ingestion, transformation, and other upstream stages.

---

## Summary of Key Python Patterns

After completing these practice problems, you should be comfortable with:

| Pattern | Key Code | Used For |
|---------|----------|----------|
| Path handling | `Path("folder/file.csv")` | Cross-platform file paths |
| File existence | `path.exists()` | Validation before operations |
| File size | `path.stat().st_size` | Upload verification |
| CSV loading | `pd.read_csv(path)` | Data inspection |
| Error handling | `try/except` | Graceful failure handling |
| Timestamps | `datetime.now().strftime("%Y/%m/%d")` | Data organization |
| Dict returns | `{"success": True, ...}` | Structured function outputs |

These patterns form the building blocks for the cloud upload functions you'll write in the lab.

---

**Next Step:** Proceed to Lab 01 - Mapping the AI Data Lifecycle